In [46]:
import unicodedata
from IPython.display import display_markdown, display, Markdown, Latex
import json
from anthropic import Anthropic, HUMAN_PROMPT, AI_PROMPT

![**I am Time**: A 19th century painting of Krishna in his _viswarūpa_. [^1]](./viswarupa.jpg){#fig-hero fig-align="center" width=80%}

## Executive Summary
You may have watched in the new Nolan movie, or read it the [2005 biography](https://archive.org/details/americanpromethe00bird/mode/1up) [@bird2005american] on which the movie is based, or indeed, or have read it in the original interview [@barnett1949j] with _Life_ magazine. Or may be you've known it all along. Julius Robert Oppenheimer is famously said to have ~~quoted~~ thought of a quote from the _Bhagavad Gitā_ upon witnessing the first nuclear explosion:
 
 > If the radiance of a thousand suns were to burst at once into the sky, that would be like the splendo[u]r of the mighty one 
 > 
 > Now I am become Death, the shatterer of worlds.

There's a lot of scholarship already published on this and indeed _Bhagavad Gita_ in itself. This post doesn't aim to add anything novel to that discussion, other than as a quick linguistic exploration through AI. 

## The Original Verses

Oppenheimer was translating Slokas 12 and 32 in Chapter 11 of the _Gita_. The verses read as follows in the original Sanskrit:

**Chapter 11, Verse 12**:

| divi sūrya-sahasrasya bhaved yugapad utthitā
| yadi bhāḥ sadṛiśhī sā syād bhāsas tasya mahātmanaḥ

**Chapter 11, Verse 32**: 

| _śhrī-bhagavān uvācha_: 
|    kālósmi loka-kṣhaya-kṛit pravṛiddho
|    lokān samāhartum iha pravṛittaḥ
|    ṛitépi tvāṁ na bhaviṣhyanti sarve
|    yévasthitāḥ pratyanīkeṣhu yodhāḥ

Historically, one learns classical Sanskrit (and classical Telugu) poetry through both a `prati padārtham` (word-by-word) translation, and a `bhāvam` (prose translation). Let's try both  here with AI.

## The Meaning

### Calling Claude 2
The advent of LLM's now mean that you can write a quick function in Python that can take in any verse in Sanskrit and return both the `prati padārtham` and the `bhāvam` in JSON. 

Here's how. 

In [64]:
#| echo: true
#| code-fold : show
def generate_sloka_meaning(sloka):
    prompt = f"""
    {HUMAN_PROMPT} Consider the following verse in Sanskrit enclosed in single quotes. Please give a JSON response with the following key/ value pairs:
    1. 'prati padartham': Aword for word translation in English in the form of a markdown table. 
    The table should have the following columns: Word in Devanagari script, Word in Telugu script, Word in Roman Script, English Translation.
    Return the table alone as the value.
    2. 'bhavam': The overall meaning of the verse in English sentences. Return the meaning alone as the value.

    Please give the response in the following JSON format: {'{"prati padartham": "table", "bhavam": "meaning"}'} This is important: do not give anything beyond the JSON response.
    The verse: '{sloka}'
    {AI_PROMPT} Can I check through my answer to ensure I have not missed anything?
    {HUMAN_PROMPT} Yes, please do. Make sure you have not missed anything. Particularly check if you're outputting the right script in the right column. 
    {AI_PROMPT}
            """
    anthropic = Anthropic()
    completion = anthropic.completions.create(
        model="claude-2",
        max_tokens_to_sample=40000,
        prompt=prompt,
    )
    return completion.completion

def extract_prati_padartham(sloka_meaning):
    #extract json value with the key 'prati padartham'
    return json.loads(sloka_meaning)["prati padartham"]

def extract_bhavam(sloka_meaning):
    #extract json value with the key 'bhavam'
    return json.loads(sloka_meaning)["bhavam"]

The function `generate_sloka_meaning` calls Claude 2 and gets it to generate a `prati padartham` and `bhavam`. The former is returned as a Markdown table, while the latter is plain text. We also got Claude 2 to check through the response before returning it to us (per suggestion in Anthropic's [docs](https://docs.anthropic.com/claude/docs/ask-claude-to-think-step-by-step)). This is because it did not generate the full meaning on some executions. 

Let's now input both verses and see what Claude 2 generates for us. 

In [65]:
#| echo: true
#| code-fold : true
#| code-overflow: wrap
verses = {
    "11.12": "divi sūryasahasrasya bhaved yugapad utthitā\nyadi bhāḥ sadṛśī sā syād bhāsas tasya mahātmanaḥ",
    "11.32": "śhrī-bhagavān uvācha\nkālo’smi loka-kṣaya-kṛt pravṛddho\nlokan samāhartum iha pravṛttaḥ\nṛte ’pi tvāṁ na bhaviṣyanti sarve\nye ’vasthitāḥ pratyanīkeṣu yodhāḥ",
}

# for each verse, generate the meaning
sloka_meanings = {}
for verse in verses:
    sloka_meanings[verse] = generate_sloka_meaning(verses[verse])

### Results

This is what we see for the two verses that Oppenheimer quoted:

In [66]:
#| echo: false
#| label: tbl-1112
#| tbl-cap: Word-for-word translation of Verse 11.12

display_markdown(f"{extract_prati_padartham(sloka_meanings['11.12'])}", raw=True)

| Word in Devanagari | Word in Telugu | Word in Roman | English Translation |
|-|-|-|-|
| दिवि | దివి | divi | in the sky |
| सूर्यसहस्रस्य | సూర్యసహస్రస్య | sūryasahasrasya | of thousands of suns |
| भवेत् | భవేత్ | bhavet | would be |
| युगपत् | యుగపత్ | yugapat | simultaneously |
| उत्थिता | ఉత్థితా | utthitā | risen |
| यदि | యది | yadi | if |
| भाः | భాः | bhāḥ | light |
| सदृशी | సదృశీ | sadṛśī | similar |
| सा | సా | sā | that |
| स्यात् | స్యాత్ | syāt | would be |
| भासस् | భాసస్ | bhāsas | the light |
| तस्य | తస్య | tasya | his |
| महात्मनः | మహాత్మనః | mahātmanaḥ | of the noble one |


In [67]:
#| echo: false
#| label: tbl-1132
#| tbl-cap: Word-for-word translation of Verse 11.32

display_markdown(f"{extract_prati_padartham(sloka_meanings['11.32'])}", raw=True)

| Word in Devanagari | Word in Telugu | Word in IAST | English Translation |
|-|-|-|-|
| श्री-भगवान् | శ్రీ-భగవాన్ | śrī-bhagavān | Lord Krishna |
| उवाच | ఉవాచ | uvāca | said |
| कालः | కాలః | kālaḥ | time |
| अस्मि | అస్మి | asmi | I am |
| लोक-क्षय-कृत् | లోక-క్షయ-కృత్ | loka-kṣaya-kṛt | the destroyer of the worlds |
| प्रवृद्धः | ప్రవృద్ధః | pravṛddhaḥ | grown |
| लोकान् | లోకాన్ | lokān | the worlds |
| समाहर्तुम् | సమాహర్తుమ్ | samāhartum | to destroy |
| इह | ఇహ | iha | here |
| प्रवृत्तः | ప్రవృత్తః | pravṛttaḥ | engaged |
| ऋते | ఋతే | ṛte | without |
| अपि | అపి | api | even |
| त्वाम् | త్వామ్ | tvām | you |
| न | న | na | not |
| भविष्यन्ति | భవిష్యన్తి | bhaviṣyanti | will be |
| सर्वे | సర్వే | sarve | all |
| ये | యే | ye | who |
| अवस्थिताः | అవస్థితాః | avasthitāḥ | arrayed |
| प्रत्यनीकेषु | ప్రత్యనీకేషు | pratyanīkeṣu | on the opposing |
| योधाः | యోధాః | yodhāḥ | warriors |


( _Do note that Claude 2 has gotten a few Roman transliterations wrong._ )

In [56]:
display(Markdown(f"Here are the meanings of both verses:\n\n>**11.12**: {extract_bhavam(sloka_meanings['11.12'])}\n>\n>**11.32**: {extract_bhavam(sloka_meanings['11.32'])}"))

Here are the meanings of both verses:

>**11.12**: If thousands of suns were to rise simultaneously in the sky, that brilliance would be similar to the radiance of that noble soul.
>
>**11.32**: In this verse, Lord Krishna reveals that He is time, the destroyer of worlds. He says that the time has come for Him to destroy the worlds, and He has set out to do so. He declares that even without you (Arjuna), all the warriors arrayed on the opposing side will cease to exist.

## Analysis
Let's compare the versions one by one.

In [81]:
#| label: tbl-comparison
#| tbl-cap: Comparing meanings for both verses
oppenheimer = {"11.12": "If the radiance of a thousand suns were to burst at once into the sky, that would be like the splendo[u]r of the mighty one",
               "11.32": "Now I am become Death, the shatterer of worlds."}
claude = {"11.12": extract_bhavam(sloka_meanings['11.12']),
          "11.32": extract_bhavam(sloka_meanings['11.32'])}

display(Markdown(f"""
| Verse | Version By | Meaning |
|:---:|:-------:|:----------------------------------------------:|
| 11.12 | Oppenheimer | {oppenheimer['11.12']} |
| 11.12 | Claude 2 | {claude['11.12']} |
| 11.32 | Oppenheimer | {oppenheimer['11.32']} |
| 11.32 | Claude 2 | {claude['11.32']} |
            """))


| Verse | Version By | Meaning |
|:---:|:-------:|:----------------------------------------------:|
| 11.12 | Oppenheimer | If the radiance of a thousand suns were to burst at once into the sky, that would be like the splendo[u]r of the mighty one |
| 11.12 | Claude 2 | If thousands of suns were to rise in the sky simultaneously, that light would be similar to the light of that noble one. |
| 11.32 | Oppenheimer | Now I am become Death, the shatterer of worlds. |
| 11.32 | Claude 2 | Lord Krishna said: I am time, the destroyer of worlds, fully grown for destroying the worlds. Now I am engaged here to destroy them. Even without you, all the warriors arrayed on the opposing side shall not exist. |
            

One may note the following: 

### 1. Light from a Noble or Mighty One
   
Oppenheimer's translation focusses on _bursting_ of light from a thousand suns, like in a nuclear explosion, while Claude 2's translation focuses on thousands of suns _rising_ in the sky ("_utthitā_"). Oppenheimer translated _mahātma_ as "mighty one", while the Claude 2 translation renders it as "noble one". 

Which is to say: while in Oppenheimer's translation, the narrator is in awe of a sudden display of immense power, in the latter, the narrator also ascribes a sense of nobility to a gentle - if less scary - display of power.

### 2. Death or Time
   
Oppenheimer translated _kālaḥ_ as "Death", while the Claude 2 translation rendered it as "Time". A quick search across multiple Sanskrit dictionaries suggests that [kālaḥ](https://andhrabharati.com/dictionary/sanskrit/index.php?w=%E0%A4%95%E0%A4%BE%E0%A4%B2) can indeed mean both "Death" and "Time", but the latter is more common. 

In particular, Monier Williams 2nd Edition (1899) tried to reconcile both meanings by suggesting that _kālaḥ_ could mean "death by age", or time as an entity that "destroys all things" [@andhrabharati2023kala]

Time has been worshipped as entity in Hinduism since the Vedic Era. Rig Veda Mandala 10, Sukta 190 suggests Time originated first from the Great Ocean in the sky, and only then did the Sun and the Moon come into existence:

In [93]:
rigveda = f"""
సముద్రాద్ అర్ణవాద్ అధి సంవత్సరో అజయాత\n
అహో రాత్రాణి విదధద్ విశ్వస్య మిషతో వశీ\n
సూర్యా చంద్రమసో ధాతా యథాపూర్వం అకల్పయత్\n
దివం చ ప్రథివీంచ అంతరిక్షం ఆతో స్వాహః\n
"""
rigveda_markdown = rigveda.replace("\n", "<br>")
display(Markdown(f" {rigveda_markdown}"))

 <br>సముద్రాద్ అర్ణవాద్ అధి సంవత్సరో అజయాత<br><br>అహో రాత్రాణి విదధద్ విశ్వస్య మిషతో వశీ<br><br>సూర్యా చంద్రమసో ధాతా యథాపూర్వం అకల్పయత్<br><br>దివం చ ప్రథివీంచ అంతరిక్షం ఆతో స్వాహః<br><br>

Yes, of course, this too needs to be translated with the function we wrote above. 

In [90]:
#| label: tbl-rigveda
#| tbl-cap: Claude 2's translation of Rig Veda Mandala 10 Sukta 190
rigveda_meaning = generate_sloka_meaning(rigveda)
display_markdown(f"{extract_prati_padartham(rigveda_meaning)}", raw=True)
display(Markdown(f"**Meaning**:\n\n>{extract_bhavam(rigveda_meaning)}"))

| Word in Devanagari | Word in Telugu | Word in Roman | English Translation |
|-|-|-|-|
| समुद्रात् | సముద్రాద్ | samudrāt | from the ocean |
| अर्णवात् | అర్ణవాద్ | arṇavāt | and from the sky |
| अधि | అధి | adhi | above |
| संवत्सर: | సంవత్సరో | saṃvatsaro | year |
| अजयात् | అజయాత | ajayāt | unceasingly |
| अहो | అహో | aho | O! |
| रात्राणि | రాత్రాణి | rātrāṇi | nights |
| विदधत् | విదధద్ | vidadhat | placing |
| विश्वस्य | విశ్వస్య | viśvasya | of the universe |
| मिषत: | మిషతో | miṣataḥ | properly |
| वशी | వశీ | vaśī | the controller |
| सूर्या: | సూర్యా | sūryāḥ | sun |
| चन्द्रमस: | చంద్రమసో | candramasaḥ | and moon |
| धाता | ధాతా | dhātā | placing |
| यथापूर्वम् | యథాపూర్వం | yathāpūrvam | as before |
| अकल्पयत् | అకల్పయత్ | akalpayat | imagined |
| दिवम् | దివం | divam | heaven |
| च | చ | ca | and |
| पृथिवीम् | ప్రథివీంచ | pṛthivīṃ | earth |
| च | చ | ca | and |
| अन्तरिक्षम् | అంతరిక్షం | antarikṣam | space |
| आत: | ఆతో | ātaḥ | then |
| स्वा: | స్వాహః | svāḥ | himself |


**Meaning**:

>The verse describes how the creator (Brahma), after bringing forth the universe consisting of the sky, heavens, earth and space between them, then created the regulating principles exemplified by the sun and moon, which go on continuously, year after year, placing day and night properly in their assigned places just as before.

_Note that the Claude 2 translation is non-deterministic; each execution has produced a different result._

Time is one of the four primary forms of Vishnu per the Vishnu Purana, the other three being matter (`pradhāna`), visible substance (`vyakta`), and spirit (`purusha`) (via [Wikipedia](https://en.wikipedia.org/wiki/Kāla))

In that sense, Krishna telling Arjuna that He is Time is not to say he _will_ destroy the world, but that both creation and destruction are part of the same process, and that Krishna is the source of both.

## Conclusion
There is no single correct interpretation of the _Gita_ or any other Sanskrit verse. The context of the narrator matters as much as the words. Oppenheimer used verses from the _Gita_ to describe _his_ feelings about a scary new power that could bring about destruction of the world. Death, clearly, was on his mind.  Another narrator (me) in a different context would interpret the same verses differently. These could very well be describing a more benevolent power over which one has no agency, but one that encompasses not just destruction but also creation. 

A quick lexical analysis through the traditional approach of seeing a word-for-word translation and then a sentence-wise translation can assist in this regard. Whilst this was traditionally done by hand, now you can script it out in Python by relying on LLM's. However, output from LLM's can be subjective too - biased by training, and also non-deterministic. You still need to think this through for yourself.

[^1]: The image is taken from [V&A's website](https://collections.vam.ac.uk/item/O154643/vishnu-as-vishvarupa-cosmic-or-painting-unknown/). It used to be shown in the India collection along with other works of art the British Raj stole from the sub-continent, but now is no longer on public display. Infuriatingly, V&A somehow asserts copyright and therefore conditions over an image of this 200-year-old painting. 